<div style="text-align:center; border-radius:20px; padding:25px; color:white; margin:0; font-family:sans-serif; background:#1b002a; box-shadow:0px 6px 18px rgba(0,0,0,0.35); margin-bottom:1em;">
          <div style="font-size:200%; color:#FEE100; font-weight:700;">
            digital-insomnia-screen-habits-sleep-ETL,EDA,Visualize,Model
          </div>
        </div>
        

In [ ]:
import os
main_csv_local_path = 'sleep_debt_screen_time.csv'
DATA_DIR = ''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == main_csv_local_path:
            DATA_DIR = os.path.join(dirname, filename)
            print(DATA_DIR)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)

# A) Load CSV from DATA_DIR
try:
    df = pd.read_csv(DATA_DIR, encoding='utf-8')
except Exception as e:
    df = pd.read_csv(DATA_DIR, encoding='latin1')

print("=== A) LOADED DATA ===")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# B) Audit shape, missing values, duplicates, numeric and categorical columns
print("\n=== B) AUDIT DATA ===")
print("Missing values per column:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
numeric_cols_raw = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols_raw = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
print(f"Numeric columns ({len(numeric_cols_raw)}): {numeric_cols_raw}")
print(f"Categorical columns ({len(categorical_cols_raw)}): {categorical_cols_raw}")

# C) Clean a copy named df_clean
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()

# D) EDA summaries & guarded distribution/correlation plots
print("\n=== D) EDA SUMMARIES ===")
print(df_clean.describe(include='all'))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
num_for_plot = numeric_cols_raw[0] if len(numeric_cols_raw) > 0 else None
if num_for_plot:
    sns.histplot(df_clean[num_for_plot].dropna(), kde=True, color='skyblue')
    plt.title(f'Distribution of {num_for_plot}')
plt.subplot(1, 2, 2)
if len(numeric_cols_raw) >= 2:
    sns.scatterplot(data=df_clean, x=numeric_cols_raw[0], y=numeric_cols_raw[1], alpha=0.5)
    plt.title(f'{numeric_cols_raw[0]} vs {numeric_cols_raw[1]}')
plt.tight_layout()
plt.show()

if len(numeric_cols_raw) > 1:
    plt.figure(figsize=(8, 6))
    corr = df_clean[numeric_cols_raw].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Matrix of Numeric Features')
    plt.tight_layout()
    plt.show()

# E) Pick TARGET_COL
TARGET_COL = 'sleep_debt_category'
if TARGET_COL not in df_clean.columns:
    TARGET_COL = None

print(f"\n=== E) TARGET COLUMN: {TARGET_COL} ===")

# F & G) Lightweight Feature Engineering (Safe Extras)
datetime_cols = []
for col in df_clean.columns:
    if 'date' in col.lower() or 'time' in col.lower() and col not in ['bedtime_hour', 'screen_time_before_bed_min']:
        try:
            converted = pd.to_datetime(df_clean[col], errors='coerce')
            if converted.notnull().sum() > 0.5 * len(df_clean):
                df_clean[col] = converted
                datetime_cols.append(col)
        except:
            pass

for col in datetime_cols:
    df_clean[f'{col}__year'] = df_clean[col].dt.year
    df_clean[col] = df_clean[col].dt.month
    df_clean[f'{col}__dayofweek'] = df_clean[col].dt.dayofweek

for col in categorical_cols_raw:
    if col != TARGET_COL:
        df_clean[f'{col}__len'] = df_clean[col].astype(str).str.len()
        df_clean[f'{col}__words'] = df_clean[col].astype(str).str.split().str.len()

# H) Problem Type Detection
def detect_problem_type(df_in, target):
    if target is None or target not in df_in.columns:
        return 'unsupervised'
    s = df_in[target]
    if s.dtype == 'object' or s.dtype.name == 'category' or s.dtype == 'bool':
        return 'classification'
    elif np.issubdtype(s.dtype, np.number):
        unique_count = s.nunique()
        if unique_count <= 15 or (unique_count / len(s) < 0.05):
            return 'classification'
        else:
            return 'regression'
    return 'classification'

problem_type = detect_problem_type(df_clean, TARGET_COL)
print(f"\n=== H) PROBLEM TYPE DETECTION ===")
print(f"Detected Problem Type: {problem_type} for target '{TARGET_COL}'")

# Helper functions for preprocessor and evaluation
def build_preprocessor(X):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, num_cols),
            ('cat', categorical_transformer, cat_cols)
        ]
    )
    return preprocessor

def evaluate_classification(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {'mae': mae, 'mse': mse, 'rmse': rmse, 'r2': r2}

def run_kmeans(X_num):
    inertias = []
    sil_scores = []
    k_range = range(2, min(9, len(X_num)))
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_num)
        inertias.append(km.inertia_)
        if len(set(labels)) > 1:
            sil_scores.append(silhouette_score(X_num, labels))
        else:
            sil_scores.append(-1)
    return k_range, inertias, sil_scores

model_results = {}
best_model = None

# I, J, K) Modeling, Evaluation & Visualization
if problem_type == 'classification':
    print("\n=== I) & J) CLASSIFICATION MODELING ===")
    drop_cols = [TARGET_COL]
    if 'user_id' in df_clean.columns:
        drop_cols.append('user_id')
    
    X = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])
    y = df_clean[TARGET_COL]
    
    valid_idx = y.notnull()
    X = X[valid_idx]
    y = y[valid_idx]
    
    stratify_arg = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify_arg)
    
    preprocessor = build_preprocessor(X)
    
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
        'KNeighborsClassifier': KNeighborsClassifier(),
        'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
        'RandomForestClassifier': RandomForestClassifier(random_state=42)
    }
    
    results_list = []
    for name, model in models.items():
        try:
            clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            metrics = evaluate_classification(y_test, y_pred)
            metrics['model_name'] = name
            metrics['pipeline'] = clf
            results_list.append(metrics)
            model_results[name] = metrics
            print(f"Successfully trained {name}: F1={metrics['f1']:.4f}")
        except Exception as e:
            print(f"Model {name} failed with error: {e}")
            
    if results_list:
        results_df = pd.DataFrame(results_list).sort_values(by='f1', ascending=False)
        best_row = results_df.iloc[0]
        best_model = best_row['pipeline']
        print("\nClassification Results Summary:")
        print(results_df[['model_name', 'accuracy', 'precision', 'recall', 'f1']])
        print(f"\nBest Model: {best_row['model_name']} with F1: {best_row['f1']:.4f}")
        
        y_pred_best = best_model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred_best)
        print("\nClassification Report:\n", classification_report(y_test, y_pred_best, zero_division=0))
        
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - {best_row["model_name"]}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()

elif problem_type == 'regression':
    print("\n=== I) & J) REGRESSION MODELING ===")
    drop_cols = [TARGET_COL]
    if 'user_id' in df_clean.columns:
        drop_cols.append('user_id')
        
    X = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])
    y = df_clean[TARGET_COL]
    
    valid_idx = y.notnull()
    X = X[valid_idx]
    y = y[valid_idx]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    preprocessor = build_preprocessor(X)
    
    models = {
        'LinearRegression': LinearRegression(),
        'KNeighborsRegressor': KNeighborsRegressor(),
        'DecisionTreeRegressor': DecisionTreeRegressor(random_state=42),
        'RandomForestRegressor': RandomForestRegressor(random_state=42)
    }
    
    results_list = []
    for name, model in models.items():
        try:
            reg = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)
            metrics = evaluate_regression(y_test, y_pred)
            metrics['model_name'] = name
            metrics['pipeline'] = reg
            results_list.append(metrics)
            model_results[name] = metrics
            print(f"Successfully trained {name}: R2={metrics['r2']:.4f}")
        except Exception as e:
            print(f"Model {name} failed with error: {e}")
            
    if results_list:
        results_df = pd.DataFrame(results_list).sort_values(by='r2', ascending=False)
        best_row = results_df.iloc[0]
        best_model = best_row['pipeline']
        print("\nRegression Results Summary:")
        print(results_df[['model_name', 'mae', 'rmse', 'r2']])
        print(f"\nBest Model: {best_row['model_name']} with R2: {best_row['r2']:.4f}")
        
        y_pred_best = best_model.predict(X_test)
        
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        sns.scatterplot(x=y_test, y=y_pred_best, alpha=0.6)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title('Actual vs Predicted')
        
        plt.subplot(1, 2, 2)
        residuals = y_test - y_pred_best
        sns.scatterplot(x=y_pred_best, y=residuals, alpha=0.6)
        plt.axhline(0, color='r', linestyle='--')
        plt.xlabel('Predicted')
        plt.ylabel('Residuals')
        plt.title('Residual Plot')
        plt.tight_layout()
        plt.show()

else:
    print("\n=== I) & J) UNSUPERVISED CLUSTERING ===")
    num_df = df_clean.select_dtypes(include=[np.number]).dropna()
    if len(num_df) > 5:
        k_range, inertias, sil_scores = run_kmeans(num_df)
        model_results['kmeans_inertias'] = inertias
        model_results['kmeans_silhouettes'] = sil_scores
        
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(list(k_range), inertias, marker='o')
        plt.title('Elbow Plot')
        plt.xlabel('k')
        plt.ylabel('Inertia')
        
        plt.subplot(1, 2, 2)
        plt.plot(list(k_range), sil_scores, marker='o', color='orange')
        plt.title('Silhouette Scores')
        plt.xlabel('k')
        plt.ylabel('Silhouette Score')
        plt.tight_layout()
        plt.show()
        
        if sil_scores and max(sil_scores) > 0:
            best_k = list(k_range)[np.argmax(sil_scores)]
        else:
            best_k = 3
        best_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
        best_model.fit(num_df)
        
        plt.figure(figsize=(6, 5))
        num_cols_list = num_df.columns[:2]
        sns.scatterplot(data=num_df, x=num_cols_list[0], y=num_cols_list[1], hue=best_model.labels_, palette='viridis', legend=False)
        plt.title(f'Clusters (k={best_k})')
        plt.tight_layout()
        plt.show()

# L) Final Artifact Output
print("\n=== L) FINAL ARTIFACT OUTPUT ===")
print("Original Shape:", df.shape)
print("Cleaned Shape:", df_clean.shape)
print("Detected Problem Type:", problem_type)
print("Models Trained:", list(model_results.keys()))
print("Best Model:", type(best_model).__name__ if best_model else None)
print("\nFirst 5 rows of df_clean:")
print(df_clean.head())
